In [1]:
import pandas as pd
import numpy as np


def clean_column_names(df):
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
    )
    return df


def standardize_text_columns(df):
    # Standardize object/string columns
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()

        # Replace common null-like strings with actual NaN
        df[col] = df[col].replace(
            ["", " ", "NA", "N/A", "na", "n/a", "null", "NULL", "none", "None"],
            np.nan
        )

    # Example standardization for common columns
    if "gender" in df.columns:
        df["gender"] = df["gender"].str.lower().replace({
            "m": "male",
            "f": "female",
            "male ": "male",
            "female ": "female"
        })

    if "country" in df.columns:
        df["country"] = df["country"].str.title()

    return df


def convert_date_columns(df):
    # Try converting columns that contain 'date'
    for col in df.columns:
        if "date" in col:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

    return df


def convert_numeric_columns(df):
    # Try converting likely numeric columns
    possible_numeric = ["age", "salary", "income", "price", "amount", "score"]
    for col in df.columns:
        if col in possible_numeric:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def fill_missing_values(df):
    # Fill numeric missing values with median
    numeric_cols = df.select_dtypes(include=["number"]).columns
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())

    # Fill categorical missing values with mode
    categorical_cols = df.select_dtypes(include=["object"]).columns
    for col in categorical_cols:
        if not df[col].mode().empty:
            df[col] = df[col].fillna(df[col].mode()[0])

    return df


def treat_outliers_iqr(df):
    # Cap outliers using IQR method for numeric columns
    numeric_cols = df.select_dtypes(include=["number"]).columns

    for col in numeric_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        if iqr == 0:
            continue

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
        df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

    return df


def main():
    input_file = "raw_dataset.csv"          # change this to your dataset file name
    output_file = "cleaned_dataset.csv"
    summary_file = "cleaning_summary.txt"

    # Load dataset
    df = pd.read_csv(input_file)

    summary = []
    summary.append("DATA CLEANING SUMMARY")
    summary.append("=" * 50)
    summary.append(f"Original shape: {df.shape}")

    # Count missing values before cleaning
    missing_before = df.isnull().sum().sum()
    summary.append(f"Missing values before cleaning: {missing_before}")

    # Remove duplicates
    duplicate_count = df.duplicated().sum()
    df = df.drop_duplicates()
    summary.append(f"Duplicate rows removed: {duplicate_count}")

    # Clean column names
    old_columns = list(df.columns)
    df = clean_column_names(df)
    new_columns = list(df.columns)
    summary.append(f"Columns renamed: {dict(zip(old_columns, new_columns))}")

    # Standardize text values
    df = standardize_text_columns(df)

    # Convert date columns
    df = convert_date_columns(df)

    # Convert numeric columns
    df = convert_numeric_columns(df)

    # Fill missing values
    df = fill_missing_values(df)

    # Outlier treatment
    df = treat_outliers_iqr(df)

    # Fix age as integer if present
    if "age" in df.columns:
        df["age"] = df["age"].round().astype("Int64")

    # Convert datetime columns to dd-mm-yyyy format for export
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].dt.strftime("%d-%m-%Y")

    missing_after = df.isnull().sum().sum()
    summary.append(f"Missing values after cleaning: {missing_after}")
    summary.append(f"Final shape: {df.shape}")
    summary.append("\nData types after cleaning:")
    summary.append(str(df.dtypes))

    # Save cleaned dataset
    df.to_csv(output_file, index=False)

    # Save summary report
    with open(summary_file, "w", encoding="utf-8") as f:
        for line in summary:
            f.write(line + "\n")

    print("Cleaning completed successfully.")
    print(f"Cleaned dataset saved as: {output_file}")
    print(f"Summary report saved as: {summary_file}")


if __name__ == "__main__":
    main()


FileNotFoundError: [Errno 2] No such file or directory: 'raw_dataset.csv'